In [1]:
import pandas as pd


In [2]:
df = pd.read_excel("CorporateSalary.xlsx")
df
df.head(2)


,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore,Salary
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6,119300
1,Director,Master's,Marketing,6.9,13,4.3,212200


In [3]:
# Split data in to X AND y

y = df['Salary']
y

X = df.drop('Salary',axis=1)
X



,Designation,Education,Department,TenureYears,ProjectsCompleted,PerformanceScore
0,Senior Analyst,Bachelor's,Data Science,8.7,12,4.6
1,Director,Master's,Marketing,6.9,13,4.3
2,Manager,Bachelor's,Engineering,3.9,14,2.7
3,Lead Consultant,Master's,Data Science,5.3,13,2.7
4,Junior Analyst,PhD,HR,3.4,15,5.0
...,...,...,...,...,...,...
115,Director,PhD,HR,0.4,1,2.3
116,Senior Manager,Master's,Data Science,0.2,0,2.1
117,Junior Analyst,Bachelor's,Engineering,0.0,1,1.9
118,Manager,Master's,Sales,0.2,2,2.0


In [4]:
# Train and Test SPlit
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_true, y_test_true = train_test_split(X,y,test_size=0.20,random_state=42)



In [5]:
# Transformation 
Numerica_Colums = ['TenureYears','ProjectsCompleted','PerformanceScore']
Ordered_Columns = ['Designation', 'Education']
One_Hot_Columns = ['Department']

Designation_Order = ['Junior Analyst', 'Senior Analyst', 'Lead Consultant', 'Manager', 'Senior Manager','Director']
Education_Order = ["Bachelor's", "Master's", 'PhD']




In [6]:
# Transformation Actual Process
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


OrdinalEncoding_01 = ("Oridnal_Encoder",OrdinalEncoder(categories=[Designation_Order, Education_Order]),Ordered_Columns,)
OneHotEncoding_01 =  ("One_Hot_Encoder",OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"),One_Hot_Columns,)
Scaling = ("Numerical_Coluns", StandardScaler(), Numerica_Colums)

transformer_engine = ColumnTransformer(transformers=[OrdinalEncoding_01,OneHotEncoding_01,Scaling]).set_output(transform="pandas")

transformer_engine



ColumnTransformer(transformers=[('Oridnal_Encoder',
                                 OrdinalEncoder(categories=[['Junior Analyst',
                                                             'Senior Analyst',
                                                             'Lead Consultant',
                                                             'Manager',
                                                             'Senior Manager',
                                                             'Director'],
                                                            ["Bachelor's",
                                                             "Master's",
                                                             'PhD']]),
                                 ['Designation', 'Education']),
                                ('One_Hot_Encoder',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Department']),
                                ('Numerical_Coluns', StandardScaler(),
                                 ['TenureYears', 'ProjectsCompleted',
                                  'PerformanceScore'])])

In [7]:
X_train_Transform = transformer_engine.fit_transform(X_train)

X_train_Transform.head(2)



,Oridnal_Encoder__Designation,Oridnal_Encoder__Education,One_Hot_Encoder__Department_Engineering,One_Hot_Encoder__Department_HR,One_Hot_Encoder__Department_Marketing,One_Hot_Encoder__Department_Product,One_Hot_Encoder__Department_Sales,Numerical_Coluns__TenureYears,Numerical_Coluns__ProjectsCompleted,Numerical_Coluns__PerformanceScore
42,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.058633,1.084800,0.063193
12,3.0,2.0,0.0,0.0,0.0,1.0,0.0,-1.049288,-0.200889,-1.030770


In [8]:
#----------------------Until This--------------------

import joblib
joblib.dump(transformer_engine,"transform.pkl")



['transform.pkl']

In [13]:
from sklearn.linear_model import ElasticNet

Elastic_Model = ElasticNet(alpha=1.0, l1_ratio=0.5)

Elastic_Model.fit(X_train_Transform,y_train_true)


ElasticNet()

In [ ]:
import joblib
joblib.dump(Elastic_Model,"Elastic_Model.pkl")



['Elastic_Model.pkl']

In [17]:
joblib.dump(transformer_engine,"transform.pkl")


['transform.pkl']

In [19]:
# Evaluation Training
# y_train_true
# y_train_pred

Elastic_Model_1 = joblib.load("Elastic_Model.pkl")

y_train_pred = Elastic_Model_1.predict(X_train_Transform)

y_train_pred



array([114539.87188213, 129582.31010077,  95302.87592913,  76217.14622336,
       172139.94014952, 122235.69502778,  75313.95368056, 129754.69771264,
       104507.36397274, 154521.0241635 , 115082.64369056,  25414.92766236,
       139230.40477835, 134730.83341953,  94527.32284024, 171288.644242  ,
       164779.33884467, 142898.81206856, 243608.99249117, 125514.60794808,
       118738.56565152, 130370.72739507,  72703.16539358,  90275.69588462,
        78402.39527279, 135457.80639378,  87974.40098062, 142272.15956865,
       107258.99167158,  63980.20908757, 215697.17185907,  89262.40519683,
       173890.84848783, 166413.75865885, 132770.87693024, 116701.07060379,
       128168.80773322, 134550.73025897, 106746.10964878, 136943.47529623,
       142151.08977374, 124921.49403093,  93940.0355279 , 108181.76239651,
       126599.78169523, 115602.58172598, 161415.35479628,  28498.00284522,
        86658.6739483 ,  85310.29699254,  31955.47621745, 130138.6985764 ,
       130173.25577619, 2

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score
import numpy as np

def linear_Metrics(x_Data,True_Value,Pred_Value):

    MSE = mean_squared_error(True_Value,Pred_Value) 
    MAE = mean_absolute_error(True_Value,Pred_Value)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(True_Value,Pred_Value)

    n = len(x_Data)
    p = x_Data.shape[1]
    A_R2 = 1-(1-R2)*(n-1)/(n-p-1)

    merticsDict = {"MSE": MSE,
                   "MAE":MAE,
                   "RMSE": RMSE,
                   "R2": R2,
                   "A_R2":A_R2}
    return merticsDict




In [21]:
# Training Phase Evaluation
linear_Metrics(X_train,y_train_true,y_train_pred)


{'MSE': 1436019602.5353115,
 'MAE': 19698.95843289218,
 'RMSE': 37894.84928767116,
 'R2': 0.7221047090462812,
 'A_R2': 0.7033701950494013}

In [ ]:
# Evaluation of Test
# X_test_transform
# y_test_pred



In [23]:
# X_test_transform

transfrom_Obj = joblib.load("transform.pkl")
X_test_transform = transfrom_Obj.fit_transform(X_test)

X_test_transform



,Oridnal_Encoder__Designation,Oridnal_Encoder__Education,One_Hot_Encoder__Department_Engineering,One_Hot_Encoder__Department_HR,One_Hot_Encoder__Department_Marketing,One_Hot_Encoder__Department_Product,One_Hot_Encoder__Department_Sales,Numerical_Coluns__TenureYears,Numerical_Coluns__ProjectsCompleted,Numerical_Coluns__PerformanceScore
44,1.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.609075,-0.778328,0.228976
47,2.0,1.0,0.0,1.0,0.0,0.0,0.0,-0.234260,-0.047087,-0.068074
4,0.0,2.0,0.0,1.0,0.0,0.0,0.0,-0.609075,-0.445946,1.417176
55,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.852705,-0.778328,-1.256274
26,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.009370,-1.044233,-0.068074
64,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.890186,-0.312993,-0.365124
73,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.590334,0.152342,-0.068074
10,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.002631,-0.246517,0.526026
40,0.0,2.0,0.0,0.0,0.0,1.0,0.0,-1.058853,0.152342,-0.216599
107,5.0,1.0,0.0,0.0,0.0,0.0,1.0,2.745521,3.077303,1.417176


In [24]:
y_test_pred = Elastic_Model_1.predict(X_test_transform)
y_test_pred



array([ 84360.17072702, 129078.82423993,  87553.40441549, 159598.9811391 ,
        70688.90133228,  85189.780783  , 163751.00978107,  71601.62101322,
        82896.84609466, 320749.43677567, 109800.28843734, 162578.62212513,
       159580.40759511, 109567.92411307,  97893.20585721, 140090.90232147,
       153526.07103878, 108013.05309365, 173687.21190126, 250819.36427652,
       120556.62155588, 105005.06926288,  71644.55697012, 146947.54382485])

In [25]:
linear_Metrics(X_test,y_test_true,y_test_pred)


{'MSE': 8122945278.986552,
 'MAE': 41524.007661256,
 'RMSE': 90127.38362443766,
 'R2': 0.5220293083330124,
 'A_R2': 0.353333770097605}